# VoxCPM2 - clone giọng tiếng Việt trên Kaggle (GPU T4 miễn phí)

**Trước khi chạy:**

1. `Settings` -> `Accelerator` -> chọn **GPU T4 x2**
2. `Settings` -> **internet đang BẬT** (menu ghi "Turn off internet" = đang bật, đúng rồi)
3. Panel phải -> `Input` -> nút **`Upload`** -> **`New Dataset`** -> tải lên `ai-dev-voice-clean.wav`

Rồi bấm `Run All`. Lần đầu tải model ~5GB.

---
### Bài học quan trọng

VoxCPM2 có 2 chế độ clone, khác nhau rất nhiều:

| Chế độ | Cách gọi | Đặc điểm |
|---|---|---|
| **Ultimate** | `prompt_wav_path` + `prompt_text` + `reference_wav_path` | Giống giọng nhất, NHƯNG sao chép luôn **nhịp nói** của mẫu và **TẮT điều khiển phong cách** |
| **Cơ bản** | chỉ `reference_wav_path` | Cho phép **điều khiển phong cách** bằng chỉ thị `(...)` ở đầu văn bản |

Notebook này dùng **chế độ cơ bản**. Nếu giọng mẫu nói nhanh, chế độ Ultimate sẽ ép bài
đọc nhanh theo và không cách nào chỉnh được.

## Ô 1 - Cài đặt

In [ ]:
!pip install -q --no-deps voxcpm==2.0.3
!pip install -q einops librosa soundfile hf_transfer
print('Cai xong.')

## Ô 2 - Tải model + nạp lên GPU

Ô này tự giải phóng model cũ và tự chọn GPU còn trống - chạy lại bao nhiêu lần cũng không tràn VRAM.

In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

import json, glob, gc, torch, numpy as np, soundfile as sf, librosa
from huggingface_hub import snapshot_download

# --- Giai phong model cu (neu chay lai o nay) ---
if 'model' in globals():
    del model
gc.collect(); torch.cuda.empty_cache()

# --- Tu chon GPU con trong nhat (T4 x2: neu GPU0 dang ban thi dung GPU1) ---
best, best_free = 0, -1
for gi in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(gi)
    print(f'  GPU{gi}: trong {free/1e9:.1f}/{total/1e9:.1f} GB')
    if free > best_free:
        best, best_free = gi, free
DEVICE = f'cuda:{best}'
assert best_free > 6e9, 'Ca 2 GPU deu day. Vao menu Run -> Restart Session roi chay lai.'

cc = torch.cuda.get_device_capability(best)
print(f'Dung {DEVICE} ({torch.cuda.get_device_name(best)}, sm_{cc[0]}{cc[1]}) | torch {torch.__version__}')

local_dir = snapshot_download('openbmb/VoxCPM2')

# T4 = sm_75, khong ho tro bf16 -> ep float16.
cfg_path = os.path.join(local_dir, 'config.json')
cfg = json.load(open(cfg_path, encoding='utf-8'))
if cc[0] < 8 and cfg.get('dtype') == 'bfloat16':
    cfg['dtype'] = 'float16'
    json.dump(cfg, open(cfg_path, 'w', encoding='utf-8'))
print('dtype:', cfg.get('dtype'))

from voxcpm import VoxCPM
model = VoxCPM.from_pretrained(local_dir, load_denoiser=False, optimize=False, device=DEVICE)
SR = model.tts_model.sample_rate
print('San sang. Sample rate:', SR)

## Ô 3 - Giọng mẫu + văn bản

Dùng **nguyên file giọng mẫu**, không xử lý gì (kéo giãn làm giọng bị méo).
`REF_TEXT` phải gõ **đúng từng chữ** lời trong file mẫu.


In [ ]:
cands = sorted(glob.glob('/kaggle/input/**/*.wav', recursive=True) +
               glob.glob('/kaggle/input/**/*.mp3', recursive=True) +
               glob.glob('/kaggle/input/**/*.MP3', recursive=True))
assert cands, 'Chua thay file giong mau. Panel phai -> Input -> Upload -> New Dataset'
REF_WAV = cands[0]          # DUNG NGUYEN file mau, khong xu ly gi
print('Giong mau:', REF_WAV)

# LOI MAU: go DUNG TUNG CHU loi thoai trong file mau.
REF_TEXT = ("Mà khoan đã, đời đâu có dễ vậy. Mà muốn đối chiếu, thì phải dựa trên dữ liệu. "
            "Mấy mô hình học trên cả núi dữ liệu, mỗi bước mà bắt nó dò lại hết cả núi đó, "
            "thì chạy tới mùa quýt mới xong, vậy ngoài thực tế người ta làm sao cho kịp đây")

TEXT = """Trên đường đạo cũng giống như trên đường đời, sẽ có nhiều thử thách. Nếu vừa gặp chút thử thách đã vội ngã lòng, mất đạo tâm, thì không thể tiến xa được. Những thử thách, những nghịch cảnh, những trở ngại, những ngang trái, những oan ức, đều là cơ hội rèn luyện đạo tâm của ta, hãy dùng những thử thách đó như món quà quý giá chứ đừng quay lưng chạy trốn."""

print(f'{len(TEXT)} ky tu')


---
## Chỉnh cho vừa tai

Ô 4 in ra **tốc độ ký tự/giây** - dùng con số đó để chỉnh, đừng mò:

| Muốn | Chỉnh |
|---|---|
| **Đọc chậm hơn** | Giảm `SLOW_REF` ở Ô 3 (0.72 -> 0.65). Cần gạt mạnh nhất |
| Đọc nhanh hơn | Tăng `SLOW_REF` (0.85) hoặc đặt `0` để dùng mẫu gốc |
| Ngắt nghỉ lâu hơn | Tăng `GAP_PARA` / `GAP_SENT` ở Ô 4 |
| Ngữ điệu khác | Đổi `STYLE` (tiếng Anh/Trung, ngắn gọn) |
| Câu dài đọc liền hơi | Giảm `MAX_CHARS` xuống 200 |
| Bám văn bản hơn | `cfg_value` 2.0 -> 2.5 hoặc 3.0 |
| Mượt hơn (chậm hơn) | `inference_timesteps` 10 -> 20 |

Model sinh **ngẫu nhiên** - chạy lại Ô 4 vài lần rồi chọn bản ưng nhất.

## Những thứ KHÔNG tồn tại (đừng tìm)

Không có tham số `speed` / `rate` / `tempo` / `duration`; không hỗ trợ SSML `<break>`;
không có token ngắt nghỉ. Xuống dòng trong văn bản **bị xóa** (mã nguồn: `text.replace("\n", " ")`).
Cách duy nhất đổi nhịp đọc là **đổi nhịp của giọng mẫu** (`SLOW_REF`).

## Lỗi thường gặp

| Triệu chứng | Xử lý |
|---|---|
| `CUDA out of memory` khi nạp model | Ô 2 đã tự dọn + tự đổi GPU. Nếu vẫn lỗi: `Run` -> `Restart Session` |
| `CUDA out of memory` khi sinh | Giảm `MAX_CHARS` xuống 200, hoặc `inference_timesteps` xuống 6 |
| Nó **đọc to câu tiếng Anh** trong ngoặc | Bạn đang ở chế độ Ultimate. Bỏ `prompt_wav_path`/`prompt_text` |
| Audio im lặng / rè toàn bộ | fp16 tràn số. Ô 2 đổi `cfg['dtype']` thành `'float32'`, chạy lại từ Ô 2 |
| Giọng bớt giống bản gốc | Đánh đổi của chế độ cơ bản. Muốn giống tối đa: thu giọng mẫu mới đọc chậm rồi quay lại Ultimate |

**Giới hạn Kaggle:** 30 **giờ** GPU/tuần (không phải 30 phút), 12 giờ/phiên,
reset thứ Bảy 00:00 UTC. Xem tại `kaggle.com/me/account`.
Đồng hồ `Session 31m / 12 hours` trên thanh công cụ là thời gian **phiên hiện tại**, không phải quota tuần.

---
## Ô cuối - BẢNG ĐIỀU KHIỂN

Chạy xong Ô 1, 2, 3 rồi chạy ô này. Từ đó chỉ cần **kéo thanh trượt và bấm ĐỌC**.

**Muốn đọc chậm rãi hơn:** tăng **Giãn lặng mẫu** (1.0 -> 2.0). Đây là cách duy nhất
làm chậm mà **không gây méo tiếng** - nó chỉ thêm im lặng giữa các từ trong file mẫu,
không đụng vào một mẫu âm thanh có tiếng nào.


In [ ]:
import ipywidgets as W
from IPython.display import display, Audio, clear_output
import re, time, inspect, numpy as np, soundfile as sf, librosa

# ============================================================================
# CHE DO ULTIMATE CLONING (prompt_wav_path + prompt_text + reference_wav_path)
# -> giong giong nhat, KHONG dung style directive, KHONG keo gian tin hieu.
#
# Doc tu ma nguon VoxCPM2:
#   reference_wav_path        -> CHAT GIONG (audio khong co loi thoai di kem)
#   prompt_wav_path + text    -> NHIP DIEU, toc do, ngu dieu (model "doc tiep" mach nay)
# => Muon doc cham: lam cho MAU nghe thong tha hon.
#
# Cach lam cham DUY NHAT khong gay meo: GIAN CAC KHOANG LANG giua tu,
# giu nguyen 100% phan co tieng (da kiem chung: 323.569 mau khac 0 giong het,
# nang luong lech 1e-13). Khac han keo gian tin hieu - cai do bam meo roi model clone lai.
# ============================================================================


def gian_lang(y, sr, factor=1.0, top_db=35, min_gap_ms=50, max_added_ms=600):
    """Gian cac khoang LANG trong y, KHONG dung toi mau co tieng."""
    y = np.ascontiguousarray(y, dtype=np.float32)
    if factor <= 1.0:
        return y.copy(), {'gaps': 0, 'added_s': 0.0}
    iv = librosa.effects.split(y, top_db=top_db, frame_length=512, hop_length=128)
    if len(iv) < 2:
        return y.copy(), {'gaps': 0, 'added_s': 0.0}
    min_gap = int(sr * min_gap_ms / 1000)
    max_added = int(sr * max_added_ms / 1000)
    pieces, added, n = [y[:iv[0][0]]], 0, 0
    for i, (s, e) in enumerate(iv):
        pieces.append(y[s:e])
        if i + 1 < len(iv):
            gap = y[e:iv[i + 1][0]]
            if len(gap) >= min_gap:
                extra = int(np.clip(int(len(gap) * factor) - len(gap), 0, max_added))
                if extra:
                    pieces.append(np.zeros(extra, dtype=y.dtype)); added += extra; n += 1
            pieces.append(gap)
    pieces.append(y[iv[-1][1]:])
    return np.concatenate(pieces), {'gaps': n, 'added_s': added / sr}


# ---------------------------------------------------------------- widgets ---
w_dil   = W.FloatSlider(value=1.0, min=1.0, max=3.0, step=0.1, description='Giãn lặng mẫu:',
                        readout_format='.1f', continuous_update=False,
                        layout=W.Layout(width='560px'), style={'description_width': '130px'})
w_max   = W.IntSlider(value=160, min=60, max=400, step=10, description='Ký tự/cụm:',
                      continuous_update=False,
                      layout=W.Layout(width='560px'), style={'description_width': '130px'})
w_gap_s = W.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.05, description='Nghỉ hết câu:',
                        readout_format='.2f', continuous_update=False,
                        layout=W.Layout(width='560px'), style={'description_width': '130px'})
w_gap_p = W.FloatSlider(value=1.10, min=0.0, max=3.0, step=0.05, description='Nghỉ hết đoạn:',
                        readout_format='.2f', continuous_update=False,
                        layout=W.Layout(width='560px'), style={'description_width': '130px'})
w_cfg   = W.FloatSlider(value=2.0, min=1.0, max=4.0, step=0.1, description='cfg_value:',
                        readout_format='.1f', continuous_update=False,
                        layout=W.Layout(width='560px'), style={'description_width': '130px'})
w_steps = W.IntSlider(value=10, min=4, max=30, step=1, description='Số bước:',
                      continuous_update=False,
                      layout=W.Layout(width='560px'), style={'description_width': '130px'})
w_text  = W.Textarea(value=TEXT, description='Văn bản:', layout=W.Layout(width='880px', height='170px'),
                     style={'description_width': '130px'})
w_go    = W.Button(description='ĐỌC', button_style='primary', icon='play',
                   layout=W.Layout(width='150px', height='42px'))
w_prev  = W.Button(description='Nghe mẫu đã giãn', icon='volume-up',
                   layout=W.Layout(width='200px', height='42px'))
w_out   = W.Output()

_cache = {}


def _mau(factor):
    """Tra ve duong dan mau da gian lang (co cache)."""
    key = round(factor, 2)
    if key <= 1.0:
        return REF_WAV, {'gaps': 0, 'added_s': 0.0}
    if key not in _cache:
        y, sr = librosa.load(REF_WAV, sr=None, mono=True)
        out, st = gian_lang(y, sr, factor=key)
        path = f'/kaggle/working/ref_gian_{int(key*10)}.wav'
        sf.write(path, out, sr)
        _cache[key] = (path, st)
    return _cache[key]


def _cum(text, limit):
    out = []
    for para in [p for p in text.replace(chr(13), '').split(chr(10)) if p.strip()]:
        cau = [s.strip() for s in re.split(r'(?<=[.!?…])\s+', para) if s.strip()]
        buf = ''
        for s in cau:
            if buf and len(buf) + len(s) + 1 > limit:
                out.append((buf, 'sent')); buf = s
            else:
                buf = (buf + ' ' + s).strip()
        if buf:
            out.append((buf, 'para'))
    return out


def _nghe_mau(_):
    with w_out:
        clear_output(wait=True)
        path, st = _mau(w_dil.value)
        y, sr = librosa.load(path, sr=None, mono=True)
        print(f'Mẫu giãn x{w_dil.value:.1f}: {len(y)/sr:.2f}s '
              f'(+{st["added_s"]:.2f}s vào {st["gaps"]} khoảng lặng)')
        print('Phần có tiếng giữ nguyên từng mẫu - chỉ thêm im lặng.')
        display(Audio(path))


def _doc(_):
    with w_out:
        clear_output(wait=True)
        txt = w_text.value.strip()
        if not txt:
            print('Chưa nhập văn bản.'); return
        w_go.disabled = True; w_go.description = 'Đang đọc...'
        try:
            ref, st = _mau(w_dil.value)
            ok = set(inspect.signature(model._generate).parameters)
            kw = {k: v for k, v in dict(cfg_value=w_cfg.value,
                                        inference_timesteps=w_steps.value,
                                        normalize=False).items() if k in ok}
            cum = _cum(txt, w_max.value)
            print(f'{len(txt)} ký tự -> {len(cum)} cụm | mẫu giãn x{w_dil.value:.1f} '
                  f'(+{st["added_s"]:.2f}s lặng) | Ultimate Cloning')
            parts, t0, tieng = [], time.time(), 0.0
            for i, (s, kind) in enumerate(cum):
                wav = np.asarray(model.generate(
                    text=s, prompt_wav_path=ref, prompt_text=REF_TEXT,
                    reference_wav_path=ref, **kw), dtype=np.float32)
                parts.append(wav); tieng += len(wav) / SR
                if i < len(cum) - 1:
                    gap = w_gap_p.value if kind == 'para' else w_gap_s.value
                    if gap > 0:
                        parts.append(np.zeros(int(gap * SR), dtype=np.float32))
                print(f'  cụm {i+1}/{len(cum)}: {len(s)} ký tự -> {len(wav)/SR:.1f}s '
                      f'({len(s)/(len(wav)/SR):.1f} ký tự/giây)')
            audio = np.concatenate(parts)
            peak = float(np.max(np.abs(audio)))
            if peak > 0.99:
                audio = audio * (0.99 / peak)
            out_path = '/kaggle/working/voxcpm2_output.wav'
            sf.write(out_path, audio, SR)
            tong = len(audio) / SR
            print(f'\nXong sau {time.time()-t0:.0f}s | {tong:.1f}s')
            print(f'  Tốc độ khi ĐANG NÓI : {len(txt)/tieng:.1f} ký tự/giây  <- chỉnh bằng Giãn lặng mẫu')
            print(f'  Tốc độ cả bài       : {len(txt)/tong:.1f} ký tự/giây  <- chỉnh thêm bằng Nghỉ hết câu/đoạn')
            print('  Tham chiếu: đạo lý/thơ ~11-13 | tin tức ~15-17')
            display(Audio(out_path, autoplay=False))
            print(f'Tải về: panel phải -> Output -> {out_path}')
        except Exception as e:
            print(f'LỖI {type(e).__name__}: {e}')
        finally:
            w_go.disabled = False; w_go.description = 'ĐỌC'


w_go.on_click(_doc)
w_prev.on_click(_nghe_mau)

display(W.VBox([
    W.HTML('<h3 style="margin:4px 0">Bảng điều khiển VoxCPM2 - Ultimate Cloning</h3>'
           '<div style="color:#666;font-size:13px;margin-bottom:8px">'
           '<b>Giãn lặng mẫu</b> = cách làm chậm KHÔNG gây méo: chỉ kéo dài các khoảng lặng '
           'giữa từ trong file mẫu, giữ nguyên từng mẫu âm thanh có tiếng. '
           'Model thấy mẫu nói thong thả hơn nên đọc chậm theo.<br>'
           'Bấm <i>Nghe mẫu đã giãn</i> để kiểm tra mẫu trước khi đọc cả bài.</div>'),
    w_dil, W.HBox([w_prev]),
    W.HTML('<div style="height:8px"></div>'),
    w_max, w_gap_s, w_gap_p, w_cfg, w_steps, w_text, w_go, w_out,
]))
